## Import Libraries


In [2]:
# Import Polars for dataframe analytics
import polars as pl
# Improt Path to manage file paths
from pathlib import Path

## Define Data Path


In [3]:
# Define data folder path
data_folder = Path("../data")

## Load Cleaned Dataset


In [4]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview Dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Create Customer-Level Metrics


In [5]:
# Aggregate customer level behavior
customer_metrics = (sales_df.group_by("customer_id").agg(

    # Total orders per customer
    pl.len().alias("total_orders"),

    # Total money spent
    pl.col("revenue").sum().alias("total_spent"),

    # Average order value
    pl.col("revenue").mean().alias("avg_order_value"),

    # First Purchase date
    pl.col("purchase_date").min().alias("first_purchase"),

    # Last purchase date
    pl.col("purchase_date").max().alias("last_purchase")

)
)
customer_metrics.head()

customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase
str,u32,i64,f64,datetime[μs],datetime[μs]
"""c970""",50,16850,337.0,2024-01-01 00:00:00,2024-12-10 00:00:00
"""c935""",73,23277,318.863014,2024-01-05 00:00:00,2024-12-31 00:00:00
"""c486""",48,15652,326.083333,2024-01-11 00:00:00,2024-11-28 00:00:00
"""c589""",69,21231,307.695652,2024-01-11 00:00:00,2024-12-30 00:00:00
"""c717""",48,15052,313.583333,2024-01-03 00:00:00,2024-12-20 00:00:00


## Calculate Customer Lifetime


In [6]:
# Calculate customer lifetime in days
customer_metrics = customer_metrics.with_columns(
    (pl.col("last_purchase") - pl.col("first_purchase")
     ).dt.total_days().alias("customer_lifetime_days")
)

customer_metrics.tail(10)

customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase,customer_lifetime_days
str,u32,i64,f64,datetime[μs],datetime[μs],i64
"""c249""",47,15153,322.404255,2024-01-21 00:00:00,2024-12-10 00:00:00,324
"""c810""",48,16502,343.791667,2024-01-05 00:00:00,2024-12-25 00:00:00,355
"""c308""",11,3389,308.090909,2024-01-27 00:00:00,2024-10-27 00:00:00,274
"""c338""",55,17595,319.909091,2024-01-01 00:00:00,2024-12-27 00:00:00,361
"""c797""",43,13707,318.767442,2024-01-06 00:00:00,2024-12-31 00:00:00,360
"""c475""",59,21291,360.864407,2024-01-08 00:00:00,2024-12-31 00:00:00,358
"""c374""",96,33154,345.354167,2024-01-06 00:00:00,2024-12-27 00:00:00,356
"""c704""",38,14012,368.736842,2024-01-04 00:00:00,2024-12-29 00:00:00,360
"""c741""",42,13858,329.952381,2024-01-20 00:00:00,2024-12-31 00:00:00,346


## Calculate Purchase Frequency


In [7]:
# Purchase frequecy = orders / lifetime
customer_metrics = customer_metrics.with_columns(
    (
        pl.col("total_orders") / (pl.col("customer_lifetime_days")+1
                                  )
    ).alias("purchase_frequency")
)

# Note:- adding +1 avoids divide by 0 for single purchase customer
customer_metrics.head()

customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency
str,u32,i64,f64,datetime[μs],datetime[μs],i64,f64
"""c970""",50,16850,337.0,2024-01-01 00:00:00,2024-12-10 00:00:00,344,0.144928
"""c935""",73,23277,318.863014,2024-01-05 00:00:00,2024-12-31 00:00:00,361,0.201657
"""c486""",48,15652,326.083333,2024-01-11 00:00:00,2024-11-28 00:00:00,322,0.148607
"""c589""",69,21231,307.695652,2024-01-11 00:00:00,2024-12-30 00:00:00,354,0.194366
"""c717""",48,15052,313.583333,2024-01-03 00:00:00,2024-12-20 00:00:00,352,0.135977


# Save Customer Level Dataset

This dataset will later power segmentation and churn analysis


In [8]:
# Save customer level metrics
customer_metrics.write_csv(data_folder / "customer_level_metrics.csv")

# Save customer level metrics in parquet
customer_metrics.write_parquet(data_folder / "customer_level_metrics.parquet")

## Calculate Global KPIs

Computing business-level metrics.


In [9]:
# Total customers
total_customers = sales_df.select(
    pl.col("customer_id").n_unique()
).item()

# Total orders
total_orders = sales_df.height

# Total revenue
total_revenue = sales_df.select(
    pl.col("revenue").sum()
).item()

## Calculate Ecommerce Metrics


In [10]:
# Average order value
aov = total_revenue / total_orders

# Purhcase frequency
purchase_frequency = total_orders / total_customers

# Revenue per customer
revenue_per_customer = total_revenue / total_customers

## Repeat Purchase Rate


In [11]:
# Customers with multiple purchases
repeat_customers = customer_metrics.filter(
    pl.col("total_orders") > 1
).height

repeat_purchase_rate = repeat_customers / total_customers

print("Repeat Customer:", repeat_customers)
print("Repeat Purchase Rate:", repeat_purchase_rate)

Repeat Customer: 1000
Repeat Purchase Rate: 1.0


## Create KPI Tables


In [12]:
# Create KPI dataframe
metrics_df = pl.DataFrame(
    {
        "metric": [
            "Total Customers",
            "Total Orders",
            "Total Revenue",
            "Average Order Value",
            "Purchase Frequency",
            "Revenue per Customer",
            "Repeat Customers",
            "Repeat Purchase Rate"
        ],
        "value": [
            float(total_customers),
            float(total_orders),
            float(total_revenue),
            round(aov, 2),
            round(purchase_frequency, 2),
            round(revenue_per_customer, 2),
            round(repeat_customers, 2),
            round(repeat_purchase_rate, 2)
        ]
    }
)

metrics_df

metric,value
str,f64
"""Total Customers""",1000.0
"""Total Orders""",50000.0
"""Total Revenue""",1.624595e7
"""Average Order Value""",324.92
"""Purchase Frequency""",50.0
"""Revenue per Customer""",16245.95
"""Repeat Customers""",1000.0
"""Repeat Purchase Rate""",1.0


In [13]:
metrics_df.sort("value", descending=True)

metric,value
str,f64
"""Total Revenue""",1.624595e7
"""Total Orders""",50000.0
"""Revenue per Customer""",16245.95
"""Total Customers""",1000.0
"""Repeat Customers""",1000.0
"""Average Order Value""",324.92
"""Purchase Frequency""",50.0
"""Repeat Purchase Rate""",1.0


In [14]:
customer_metrics.sort("total_orders", descending=True).head()

customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency
str,u32,i64,f64,datetime[μs],datetime[μs],i64,f64
"""c102""",110,36590,332.636364,2024-01-14 00:00:00,2024-12-28 00:00:00,349,0.314286
"""c22""",109,35191,322.853211,2024-01-04 00:00:00,2024-12-26 00:00:00,357,0.304469
"""c821""",104,34296,329.769231,2024-01-04 00:00:00,2024-12-30 00:00:00,361,0.287293
"""c860""",103,34947,339.291262,2024-01-07 00:00:00,2024-12-29 00:00:00,357,0.287709
"""c937""",101,32849,325.237624,2024-01-01 00:00:00,2024-12-31 00:00:00,365,0.275956


In [15]:
customer_metrics.describe()

statistic,customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency
str,str,f64,f64,f64,str,str,f64,f64
"""count""","""1000""",1000.0,1000.0,1000.0,"""1000""","""1000""",1000.0,1000.0
"""null_count""","""0""",0.0,0.0,0.0,"""0""","""0""",0.0,0.0
"""mean""",null,50.0,16245.95,324.785374,"""2024-01-11 07:32:09.600000""","""2024-12-21 17:00:57.600000""",345.395,0.143393
"""std""",null,17.516702,5803.458224,20.911029,null,null,17.474162,0.047757
"""min""","""c0""",7.0,2493.0,260.538462,"""2024-01-01 00:00:00""","""2024-10-23 00:00:00""",205.0,0.033981
"""25%""",null,38.0,12214.0,310.111111,"""2024-01-03 00:00:00""","""2024-12-18 00:00:00""",339.0,0.109145
"""50%""",null,49.0,15955.0,324.892857,"""2024-01-07 00:00:00""","""2024-12-26 00:00:00""",350.0,0.142012
"""75%""",null,61.0,19691.0,339.0,"""2024-01-15 00:00:00""","""2024-12-29 00:00:00""",358.0,0.173021
"""max""","""c999""",110.0,36590.0,394.833333,"""2024-06-06 00:00:00""","""2024-12-31 00:00:00""",365.0,0.314286


In [16]:
customer_metrics.select([
    pl.col("total_orders").min().alias("min_orders"),
    pl.col("total_orders").mean().alias("avg_orders"),
    pl.col("total_orders").max().alias("max_orders")
])

min_orders,avg_orders,max_orders
u32,f64,u32
7,50.0,110


## Save Metrics Dataset


In [17]:
# Save KPI Dataset
metrics_df.write_csv(data_folder / "customer_behavior_metrics.csv")

metrics_df.write_parquet(data_folder / "customer_behavior_metrics.parquet")

# Adding Customer Segmentation

In [36]:
customer_metrics = customer_metrics.with_columns(

    pl.when(pl.col("total_spent") > 30000).then(pl.lit("VIP"))
    .when(pl.col("total_orders") >= 30).then(pl.lit("Loyal"))
    .when(pl.col("total_orders") >= 10).then(pl.lit("Regular"))
    .otherwise(pl.lit("Low Value"))
    .alias("customer_type")

)

In [37]:
customer_metrics.group_by("customer_type").count()

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_20912\2212270806.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  customer_metrics.group_by("customer_type").count()


customer_type,count
str,u32
"""Regular""",111
"""Low Value""",2
"""Loyal""",874
"""VIP""",13


In [38]:
sales_encriched = sales_df.join(
    customer_metrics.select(["customer_id", "customer_type"]),
    on="customer_id",
    how="left"
)

In [39]:
sales_encriched.select("customer_type").unique()

customer_type
str
"""Loyal"""
"""Low Value"""
"""VIP"""
"""Regular"""


In [40]:
sales_encriched.write_parquet(data_folder / "sales_encriched.parquet")